# Step 6: Perbaikan Pipeline (Tugas #3)

Notebook ini menjalankan eksperimen E0-E3 untuk Tugas #3 (Improving the Data Science Pipeline),
menggunakan baseline pipeline terkoreksi dari Tugas #2 (lihat `AUDIT_METODOLOGI.md`, notebook
`04_preprocessing_split.ipynb` dan `05_modeling_evaluation.ipynb`; lihat juga
`03_pure_replication.ipynb` untuk versi replikasi murni tanpa koreksi anti-leakage).

Brief Tugas #3 secara eksplisit menyebut 8 kategori improvement yang bisa dipilih: missing-value
handling, categorical-variable encoding, numerical transformations, scaling/normalization, outlier
handling, feature engineering, feature selection, removal of irrelevant/redundant features. Dua
improvement di notebook ini dipilih dari daftar tersebut:

- **Improvement A** = *numerical transformations* (log1p pada fitur numerik yang skewed).
- **Improvement B** = *removal of irrelevant/redundant features* (drop fitur berkorelasi tinggi).

Model utama yang dibawa sepanjang eksperimen: **XGBoost** (performa terbaik di Tugas #2, dan yang
jadi rujukan utama di paper acuan).

Metrik utama pembanding: **MCC (Matthews Correlation Coefficient)** -- dipilih (revisi final
17 Sep 2026, setelah diskusi dengan user) karena MCC adalah satu skalar yang dihitung langsung dari
confusion matrix, robust terhadap imbalance kelas, dan TIDAK punya varian "weighted vs kelas-positif"
yang bisa dipilih-pilih setelah melihat hasil. Sempat dicoba F1-Score (weighted) sebagai primary
metric supaya sebanding dengan paper, tapi itu terbukti rawan bias: pipeline yang menang di
F1-weighted bisa jadi cuma menang lewat trade-off precision naik/recall turun yang sepihak, bukan
perbaikan riil (lihat contoh E2 di bagian diskusi). MCC dipakai sebagai kriteria seleksi/penentu
kesimpulan "membantu atau tidak" di semua tahap (E0-E3, tuning, evaluasi akhir).

**Precision (weighted) dan F1-Score (weighted) tetap ditampilkan di semua tabel**, tapi HANYA untuk
konteks "seberapa dekat pipeline ke angka yang dilaporkan paper" (paper diduga melaporkan weighted
average, bukan skor kelas positif murni -- lihat notebook `05_modeling_evaluation.ipynb`) -- bukan
untuk menentukan pipeline mana yang "menang".

Semua langkah preprocessing (encoding, log-transform, seleksi fitur chi2, scaling) di-fit ULANG di
setiap fold train lewat `CorrectedPreprocessor` (lihat `src/pipeline_experiments.py`), memastikan
tidak ada leakage antar fold -- sesuai requirement eksplisit Tugas #3 ("must be fitted only on
training data"). SMOTE (sampling_strategy='auto', 1:1) dipakai KONSTAN di semua eksperimen --
bukan lagi variabel improvement yang dieksplorasi, karena resampling bukan bagian dari 8 kategori
resmi brief.

CV: Stratified 5-fold, `random_state=42`.


In [1]:
import sys, os
sys.path.append('../src')
from data_utils import load_data
from preprocessing import split_raw
from pipeline_experiments import (
    build_pipeline, cross_validate_pipeline, REDUNDANT_FEATURES_TO_DROP, CHI2_K,
)
from models import get_models
import pandas as pd

os.makedirs('../results/tables', exist_ok=True)
pd.set_option('display.width', 120)

CV_FOLDS = 5

df = load_data()

# PENTING: gunakan split raw 70/30 IDENTIK dengan Assignment #2 (random_state=42, stratified),
# supaya perbandingan tetap adil ("use the same ... data split ... throughout this assignment").
# Semua eksperimen E0-E3 dan hyperparameter tuning (step 2-5 brief) HANYA memakai X_train/y_train
# lewat stratified k-fold CV. X_test/y_test TIDAK disentuh sampai evaluasi akhir (step 6-7),
# supaya test set benar-benar tidak ikut memengaruhi keputusan improvement mana pun.
X_train_raw, X_test_raw, y_train, y_test = split_raw(df)

xgb = get_models()['XGBoost']

print('Train (dipakai utk semua CV E0-E3 & tuning):', X_train_raw.shape)
print('Test  (disimpan, TIDAK disentuh sampai step akhir):', X_test_raw.shape)
print('Proporsi kelas positif -- train:', round(y_train.mean(), 4), '| test:', round(y_test.mean(), 4))
print('Model utama: XGBoost (max_depth=2, sesuai parameter Tugas #2, tidak diubah di sini)')

# Alias X, y -> dipakai oleh sel-sel CV di bawah (E0-E3), supaya CV hanya jalan di data train.
X, y = X_train_raw, y_train


Train (dipakai utk semua CV E0-E3 & tuning): (8631, 17)
Test  (disimpan, TIDAK disentuh sampai step akhir): (3699, 17)
Proporsi kelas positif -- train: 0.1548 | test: 0.1546
Model utama: XGBoost (max_depth=2, sesuai parameter Tugas #2, tidak diubah di sini)


**Catatan metodologi penting:** semua eksperimen E0-E3 di bawah, serta hyperparameter tuning
(step berikutnya), HANYA dievaluasi lewat stratified 5-fold CV pada porsi **train** (70%) dari
split raw yang identik dengan Assignment #2. Test set (30%) baru dipakai sekali di evaluasi akhir
(bagian "Evaluasi Pipeline Final di Test Set"), supaya keputusan improvement mana pun tidak
"melihat" test set sama sekali -- konsisten dengan requirement Tugas #3 dan audit anti-leakage
Tugas #2.


## Temuan EDA yang Melandasi Improvement (Ringkasan Evidence)

- **Fitur numerik sangat skewed**: semua 10 fitur numerik punya skewness tinggi (1,96 s.d. 7,58),
  beberapa dengan proporsi nol besar (`Informational_Duration` 80,5% nol, `PageValues` 77,9% nol,
  `SpecialDay` 89,9% nol). Distribusi yang sangat skewed berpotensi menyulitkan model membedakan
  pola di rentang nilai kecil, dan bisa membuat langkah scaling (StandardScaler) kurang efektif
  karena outlier ekstrem mendominasi mean/std.
- **Fitur redundan**: `eda_high_correlation_pairs.csv` menunjukkan `BounceRates`-`ExitRates`
  r=0,91 dan `ProductRelated`-`ProductRelated_Duration` r=0,86 -- pasangan fitur yang hampir
  mengukur hal yang sama, potensi redundansi yang tidak menambah informasi baru untuk model.

**Hipotesis (EDA finding -> proposed change -> expected effect):**

1. EDA finding: fitur numerik sangat skewed (skew 1,96-7,58), beberapa zero-inflated. Proposed
   change (Improvement A -- *numerical transformations*): terapkan log1p() pada semua fitur
   numerik sebelum seleksi chi2 dan scaling. log1p (bukan log biasa) dipilih karena banyak nilai 0
   di dataset ini, dan semua fitur numerik non-negatif secara alami sehingga log1p selalu
   terdefinisi. Expected effect: distribusi lebih mendekati normal, StandardScaler jadi lebih
   representatif, model bisa membedakan pola di rentang nilai kecil dengan lebih baik -> F1-weighted
   naik.
2. EDA finding: dua pasangan fitur numerik berkorelasi sangat tinggi (r>0,85). Proposed change
   (Improvement B -- *removal of irrelevant/redundant features*): drop satu fitur dari tiap
   pasangan (fitur dengan skor chi2 lebih rendah di notebook 04: `ExitRates` dan `ProductRelated`).
   Expected effect: mengurangi redundansi input, model XGBoost (yang cukup robust terhadap
   multikolinearitas) diperkirakan performanya minimal berubah atau sedikit membaik karena less
   noise dari fitur yang informasinya duplikat.

**Catatan revisi metodologi (17 Sep 2026):** draft eksperimen sebelumnya memakai perubahan rasio
SMOTE sebagai "Improvement A", tapi setelah membaca ulang brief lebih teliti, SMOTE/resampling
BUKAN bagian dari 8 kategori improvement resmi yang disebutkan brief (missing-value handling,
categorical encoding, numerical transformations, scaling/normalization, outlier handling, feature
engineering, feature selection, removal of redundant features) -- resampling adalah teknik
imbalance handling, bukan preprocessing dalam pengertian yang brief maksudkan. Improvement A
diganti ke log-transform numerik (kategori "numerical transformations") agar konsisten dengan
daftar 8 opsi brief. SMOTE tetap dipakai di semua eksperimen dengan `sampling_strategy='auto'`
konstan, bukan lagi variabel yang dieksplorasi.


## E0 -- Baseline (Pipeline Terkoreksi Tugas #2, Dievaluasi via 5-Fold CV)

Pipeline identik dengan Tugas #2 (chi2 top-20 -> scaling -> SMOTE auto/1:1 -> XGBoost), bedanya di
sini dievaluasi dengan stratified 5-fold CV (bukan single 70/30 split) supaya bisa dibandingkan
secara adil dengan E1-E3 yang juga pakai CV yang sama.


In [2]:
results_store = {}

pipe_e0 = build_pipeline(xgb, k=CHI2_K, drop_features=None, log_transform=False)
summary_e0, _ = cross_validate_pipeline(pipe_e0, X, y, cv_folds=CV_FOLDS)
results_store['E0 (Baseline)'] = summary_e0['Mean']

print('E0 -- Baseline (tanpa log-transform, semua 20 fitur chi2):')
print(summary_e0)


E0 -- Baseline (tanpa log-transform, semua 20 fitur chi2):
                        Mean     Std
Metric                              
Accuracy              0.8940  0.0069
Precision             0.6343  0.0217
TPR (Recall)          0.7470  0.0276
F1-Score              0.6857  0.0190
TNR                   0.9209  0.0075
MCC                   0.6257  0.0228
auROC                 0.9265  0.0076
auPR                  0.7307  0.0233
Precision (weighted)  0.9029  0.0060
F1-Score (weighted)   0.8974  0.0064


## E1 -- Improvement A: Log-Transform Fitur Numerik (Numerical Transformations)

Terapkan log1p() pada semua fitur numerik SEBELUM seleksi chi2 dan scaling. Fitur lain (kategorikal
hasil encoding, kode-kode integer) tidak ditransformasi. SMOTE dan seleksi fitur tetap sama seperti
E0 (auto/1:1, top-20 chi2) supaya efek log-transform bisa diisolasi dari perubahan lain.


In [3]:
pipe_e1 = build_pipeline(xgb, k=CHI2_K, drop_features=None, log_transform=True)
summary_e1, _ = cross_validate_pipeline(pipe_e1, X, y, cv_folds=CV_FOLDS)
results_store['E1 (Improvement A: log-transform)'] = summary_e1['Mean']

print('E1 -- Improvement A (log1p pada fitur numerik):')
print(summary_e1)


E1 -- Improvement A (log1p pada fitur numerik):
                        Mean     Std
Metric                              
Accuracy              0.8877  0.0089
Precision             0.6114  0.0283
TPR (Recall)          0.7620  0.0325
F1-Score              0.6777  0.0200
TNR                   0.9108  0.0120
MCC                   0.6168  0.0242
auROC                 0.9257  0.0075
auPR                  0.7284  0.0249
Precision (weighted)  0.9013  0.0065
F1-Score (weighted)   0.8926  0.0077


## E2 -- Improvement B: Pruning Fitur Redundan (Removal of Redundant Features)

Drop `ExitRates` dan `ProductRelated` (skor chi2 lebih rendah dari pasangannya di notebook 04),
pertahankan `BounceRates` dan `ProductRelated_Duration`. Tanpa log-transform (isolasi efek B dari A).


In [4]:
pipe_e2 = build_pipeline(xgb, k=CHI2_K, drop_features=REDUNDANT_FEATURES_TO_DROP, log_transform=False)
summary_e2, _ = cross_validate_pipeline(pipe_e2, X, y, cv_folds=CV_FOLDS)
results_store['E2 (Improvement B: drop redundant features)'] = summary_e2['Mean']

print('E2 -- Improvement B (drop ExitRates, ProductRelated):')
print(summary_e2)


E2 -- Improvement B (drop ExitRates, ProductRelated):
                        Mean     Std
Metric                              
Accuracy              0.8946  0.0095
Precision             0.6382  0.0301
TPR (Recall)          0.7395  0.0171
F1-Score              0.6850  0.0241
TNR                   0.9230  0.0085
MCC                   0.6247  0.0290
auROC                 0.9248  0.0067
auPR                  0.7227  0.0239
Precision (weighted)  0.9024  0.0075
F1-Score (weighted)   0.8977  0.0087


## E3 -- Kombinasi (Improvement A + B)

Gabungkan log-transform (A) dengan feature pruning (B).


In [5]:
pipe_e3 = build_pipeline(xgb, k=CHI2_K, drop_features=REDUNDANT_FEATURES_TO_DROP, log_transform=True)
summary_e3, _ = cross_validate_pipeline(pipe_e3, X, y, cv_folds=CV_FOLDS)
results_store['E3 (A + B combined)'] = summary_e3['Mean']

print('E3 -- Kombinasi A+B:')
print(summary_e3)


E3 -- Kombinasi A+B:
                        Mean     Std
Metric                              
Accuracy              0.8918  0.0084
Precision             0.6247  0.0228
TPR (Recall)          0.7545  0.0251
F1-Score              0.6835  0.0233
TNR                   0.9169  0.0060
MCC                   0.6230  0.0283
auROC                 0.9248  0.0078
auPR                  0.7227  0.0281
Precision (weighted)  0.9024  0.0075
F1-Score (weighted)   0.8958  0.0080


## Tabel Ringkasan E0-E3


In [6]:
comparison_e0_e3 = pd.DataFrame(results_store).T
comparison_e0_e3 = comparison_e0_e3[['Accuracy', 'Precision', 'TPR (Recall)', 'F1-Score', 'TNR', 'MCC', 'auROC', 'auPR', 'Precision (weighted)', 'F1-Score (weighted)']]
comparison_e0_e3.to_csv('../results/tables/tugas3_e0_e3_comparison.csv')
comparison_e0_e3


Metric,Accuracy,Precision,TPR (Recall),F1-Score,TNR,MCC,auROC,auPR,Precision (weighted),F1-Score (weighted)
E0 (Baseline),0.8940,0.6343,0.7470,0.6857,0.9209,0.6257,0.9265,0.7307,0.9029,0.8974
E1 (Improvement A: log-transform),0.8877,0.6114,0.7620,0.6777,0.9108,0.6168,0.9257,0.7284,0.9013,0.8926
E2 (Improvement B: drop redundant features),0.8946,0.6382,0.7395,0.6850,0.9230,0.6247,0.9248,0.7227,0.9024,0.8977
E3 (A + B combined),0.8918,0.6247,0.7545,0.6835,0.9169,0.6230,0.9248,0.7227,0.9024,0.8958


## Ringkasan E0-E3 dan Keputusan Kombinasi yang Dibawa ke Tuning

Hasil CV train-only (5-fold, MCC sebagai primary metric):

| Eksperimen | MCC | vs E0 | F1-Score (weighted, konteks) |
|---|---|---|---|
| E0 (Baseline) | 0,6257 | -- | 0,8974 |
| E1 (Improvement A: log-transform numerik) | 0,6168 | -0,0089 (lebih buruk) | 0,8926 |
| E2 (Improvement B: drop fitur redundan) | 0,6247 | -0,0010 (lebih buruk, tipis) | 0,8977 |
| E3 (A+B kombinasi) | 0,6230 | -0,0027 (lebih buruk) | 0,8958 |

Dengan MCC sebagai primary metric, kesimpulannya bersih dan konsisten: KETIGA eksperimen (E1, E2,
E3) lebih buruk dari baseline -- tidak ada yang ambigu seperti saat F1-weighted dipakai (yang
membuat E2 tampak "nyaris netral/positif" padahal MCC menunjukkan E2 tetap lebih buruk, hanya
selisihnya kecil). E3 (kombinasi) tetap dibawa ke tahap hyperparameter tuning, sesuai skema
eksperimen "selected combination" yang diminta brief -- lepas dari hasil individualnya, supaya
konsisten dengan rencana yang sudah disepakati sebelum eksperimen dijalankan.


## Step 5 -- Hyperparameter Tuning (XGBoost)

**Apakah paper sudah melakukan tuning yang adekuat?** Paper hanya menguji `max_depth` di
{2, 5, 10} untuk XGBoost (grid 1-dimensi, univariat), tanpa menyentuh `learning_rate` atau
`n_estimators` (dibiarkan default). Ini tuning yang cukup terbatas -- jadi ada ruang wajar untuk
tuning tambahan tanpa melanggar batasan brief soal "extensive brute-force searching is not
required".

**Search space yang dipakai (kecil, dijustifikasi, bukan grid brute-force):**
- `max_depth` in {2, 3, 4}: berpusat di sekitar nilai optimal paper (2), menambah 2 tetangga
  untuk cek apakah pohon sedikit lebih dalam membantu tanpa overfitting berlebih (dataset besar,
  fitur relatif sedikit setelah seleksi -- risiko overfit pohon dalam kecil tapi tetap dibatasi).
- `learning_rate` in {0.05, 0.1, 0.3}: 0.3 adalah default XGBoost, 0.1 dan 0.05 mengecek apakah
  learning rate lebih kecil (dengan estimator lebih banyak) memberi generalisasi lebih baik --
  parameter yang paper sama sekali tidak coba.
- `n_estimators` in {50, 100, 200}: rentang wajar di sekitar default (100), tidak terlalu ekstrem.

Total 27 kombinasi (3x3x3) -- kecil, jauh dari brute-force. CV: stratified 5-fold pada data TRAIN
saja (test set tidak disentuh). Metrik seleksi: MCC, konsisten dengan primary metric E0-E3.


In [7]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import make_scorer, matthews_corrcoef

pipe_for_tuning = build_pipeline(
    xgb, k=CHI2_K, drop_features=REDUNDANT_FEATURES_TO_DROP, log_transform=True,
)

param_grid = {
    'clf__max_depth': [2, 3, 4],
    'clf__learning_rate': [0.05, 0.1, 0.3],
    'clf__n_estimators': [50, 100, 200],
}

skf_tuning = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=42)
mcc_scorer = make_scorer(matthews_corrcoef)

grid_search = GridSearchCV(
    pipe_for_tuning, param_grid, scoring=mcc_scorer, cv=skf_tuning, n_jobs=-1, refit=True,
)
grid_search.fit(X_train_raw, y_train)  # X_train_raw/y_train saja, TIDAK pernah lihat test set

print('Kombinasi hyperparameter terbaik:', grid_search.best_params_)
print('MCC CV terbaik (train-only, 5-fold):', round(grid_search.best_score_, 4))


Kombinasi hyperparameter terbaik: {'clf__learning_rate': 0.1, 'clf__max_depth': 3, 'clf__n_estimators': 100}
MCC CV terbaik (train-only, 5-fold): 0.6358


In [8]:
cv_results_df = pd.DataFrame(grid_search.cv_results_)
top10 = cv_results_df.sort_values('rank_test_score').head(10)[
    ['param_clf__max_depth', 'param_clf__learning_rate', 'param_clf__n_estimators',
     'mean_test_score', 'std_test_score', 'rank_test_score']
]
top10.to_csv('../results/tables/tugas3_gridsearch_top10.csv', index=False)
top10


,param_clf__max_depth,param_clf__learning_rate,param_clf__n_estimators,mean_test_score,std_test_score,rank_test_score
13,3,0.10,100,0.635772,0.025673,1
5,3,0.05,200,0.631847,0.028466,2
11,2,0.10,200,0.631253,0.021778,3
12,3,0.10,50,0.625893,0.024950,4
14,3,0.10,200,0.625494,0.022807,5
18,2,0.30,50,0.624417,0.025323,6
4,3,0.05,100,0.624212,0.025389,7
10,2,0.10,100,0.624161,0.024720,8
2,2,0.05,200,0.623348,0.025892,9
3,3,0.05,50,0.623212,0.021241,10


## Step 6 -- Evaluasi Pipeline Final di Test Set

Sekarang, dan HANYA sekarang, test set (30%, disimpan murni sejak awal notebook) dipakai untuk
evaluasi akhir. EMPAT pipeline dibandingkan pada test set yang sama, sebagai satu rangkaian:

0. **Replikasi Murni dari Paper** (`03_pure_replication.ipynb`): chi2+scaling di-fit ke SELURUH
   dataset sebelum split (meniru kemungkinan cara kerja paper apa adanya, tanpa koreksi apa pun).
   Titik referensi paling awal -- BUKAN starting point Tugas #3, melainkan konteks historis untuk
   menunjukkan seberapa besar/kecil dampak koreksi metodologi Tugas #2.
1. **Baseline (Terkoreksi, Assignment #2)**: chi2+scaling di-fit HANYA di train (anti-leakage),
   parameter model default paper (`max_depth=2`). Ini starting point resmi Tugas #3.
2. **Improved preprocessing**: E3 (log-transform + drop 2 fitur redundan), model masih default
   paper.
3. **Final tuned**: E3 + hyperparameter terbaik dari grid search di atas.

**Penting:** Tugas #3 adalah lanjutan dari pipeline TERKOREKSI (baris 1), bukan dari replikasi
murni (baris 0). Baris 0 hanya ditampilkan sebagai konteks pembanding tambahan, tidak menjadi
bagian dari rangkaian improvement E0-E3.


In [9]:
from evaluation import evaluate_model
from sklearn.metrics import precision_score, f1_score

def fit_predict_eval(pipeline, X_tr, y_tr, X_te, y_te):
    pipeline.fit(X_tr, y_tr)
    y_pred = pipeline.predict(X_te)
    y_proba = pipeline.predict_proba(X_te)[:, 1]
    metrics = evaluate_model(y_te, y_pred, y_proba)
    # Tambahan: varian weighted (kedua kelas), untuk dibandingkan langsung dengan angka paper
    metrics['Precision (weighted)'] = precision_score(y_te, y_pred, average='weighted', zero_division=0)
    metrics['F1-Score (weighted)'] = f1_score(y_te, y_pred, average='weighted', zero_division=0)
    return metrics

# 0. Replikasi Murni dari Paper (konteks pembanding, BUKAN bagian rangkaian E0-E3)
pure_results = pd.read_csv('../results/tables/pure_replication_model_results.csv', index_col=0)
metrics_pure = pure_results.loc['XGBoost'].to_dict()

# 1. Baseline (identik Assignment #2, TERKOREKSI -- starting point resmi Tugas #3)
pipe_baseline_final = build_pipeline(xgb, k=CHI2_K, drop_features=None, log_transform=False)
metrics_baseline = fit_predict_eval(pipe_baseline_final, X_train_raw, y_train, X_test_raw, y_test)

# 2. Improved preprocessing (E3: log-transform + feature pruning), model params default paper
pipe_improved_final = build_pipeline(xgb, k=CHI2_K, drop_features=REDUNDANT_FEATURES_TO_DROP, log_transform=True)
metrics_improved = fit_predict_eval(pipe_improved_final, X_train_raw, y_train, X_test_raw, y_test)

# 3. Final tuned (E3 preprocessing + best hyperparameters)
best_params_clean = {k.replace('clf__', ''): v for k, v in grid_search.best_params_.items()}
xgb_tuned = xgb.__class__(**{**xgb.get_params(), **best_params_clean})
pipe_final_tuned = build_pipeline(xgb_tuned, k=CHI2_K, drop_features=REDUNDANT_FEATURES_TO_DROP, log_transform=True)
metrics_tuned = fit_predict_eval(pipe_final_tuned, X_train_raw, y_train, X_test_raw, y_test)

final_comparison = pd.DataFrame({
    'Replikasi Murni dari Paper (konteks)': metrics_pure,
    'Baseline Terkoreksi (Assignment #2)': metrics_baseline,
    'Improved Preprocessing (E3)': metrics_improved,
    'Final Tuned Pipeline': metrics_tuned,
}).T
final_comparison = final_comparison[['Accuracy', 'Precision', 'TPR (Recall)', 'F1-Score', 'TNR', 'MCC', 'auROC', 'auPR', 'Precision (weighted)', 'F1-Score (weighted)']].round(4)
final_comparison.to_csv('../results/tables/tugas3_final_test_evaluation.csv')
print('Evaluasi pada TEST SET (30%, baru disentuh sekarang):')
final_comparison


Evaluasi pada TEST SET (30%, baru disentuh sekarang):


,Accuracy,Precision,TPR (Recall),F1-Score,TNR,MCC,auROC,auPR,Precision (weighted),F1-Score (weighted)
Replikasi Murni dari Paper (konteks),0.8835,0.6035,0.7185,0.6560,0.9137,0.5898,0.9225,0.7037,0.8936,0.8875
Baseline Terkoreksi (Assignment #2),0.8854,0.6091,0.7220,0.6608,0.9153,0.5955,0.9194,0.6956,0.8951,0.8892
Improved Preprocessing (E3),0.8835,0.6011,0.7325,0.6604,0.9111,0.5951,0.9180,0.6843,0.8952,0.8880
Final Tuned Pipeline,0.8737,0.5688,0.7587,0.6502,0.8948,0.5840,0.9210,0.7055,0.8936,0.8808


In [10]:
print('Perbandingan F1-Score (weighted) dan Precision (weighted) dengan paper (XGBoost):')
paper_precision_w, paper_f1_w = 0.9001, 0.898  # dilaporkan paper, diduga weighted avg (lihat notebook 05)
for name in final_comparison.index:
    pw = final_comparison.loc[name, 'Precision (weighted)']
    fw = final_comparison.loc[name, 'F1-Score (weighted)']
    print(f"  {name:<38} Precision(w)={pw:.4f} (selisih {pw-paper_precision_w:+.4f})  "
          f"F1(w)={fw:.4f} (selisih {fw-paper_f1_w:+.4f})")


Perbandingan F1-Score (weighted) dan Precision (weighted) dengan paper (XGBoost):
  Replikasi Murni dari Paper (konteks)   Precision(w)=0.8936 (selisih -0.0065)  F1(w)=0.8875 (selisih -0.0105)
  Baseline Terkoreksi (Assignment #2)    Precision(w)=0.8951 (selisih -0.0050)  F1(w)=0.8892 (selisih -0.0088)
  Improved Preprocessing (E3)            Precision(w)=0.8952 (selisih -0.0049)  F1(w)=0.8880 (selisih -0.0100)
  Final Tuned Pipeline                   Precision(w)=0.8936 (selisih -0.0065)  F1(w)=0.8808 (selisih -0.0172)


## Step 7 -- Analisis dan Diskusi Hasil

**Perubahan preprocessing mana yang membantu, mana yang tidak?**

TIDAK ADA yang membantu -- baik Improvement A (log-transform numerik) maupun Improvement B (drop
fitur redundan) tidak memberi peningkatan yang berarti, baik di CV train-only maupun di test set.
Hyperparameter tuning tambahan JUSTRU memperburuk hasil di test set secara signifikan.

Tabel evaluasi test set (MCC sebagai primary metric; Precision/F1-weighted ditampilkan sebagai
konteks pembanding ke paper, bukan penentu kesimpulan):

| Pipeline | MCC (primary) | F1-Score (weighted, konteks) | Precision (weighted, konteks) |
|---|---|---|---|
| Replikasi Murni dari Paper (konteks) | 0,5898 | 0,8875 | 0,8936 |
| **Baseline Terkoreksi (Assignment #2)** | **0,5955** | 0,8892 | 0,8951 |
| Improved Preprocessing (E3) | 0,5951 | 0,8880 | 0,8952 |
| Final Tuned Pipeline | 0,5840 | 0,8808 | 0,8936 |
| Paper | -- | 0,898 | 0,9001 |

Baseline Terkoreksi (starting point resmi Tugas #3) punya MCC TERBAIK dari keempat pipeline.
Improved Preprocessing (E3) sedikit di bawahnya (-0,0004), selisih ini masuk noise CV (std MCC
sekitar 0,02-0,03). Final Tuned Pipeline JAUH lebih buruk (-0,0115 dari baseline) -- penurunan
yang cukup besar untuk tidak dianggap sekadar noise.

**Catatan metodologi metrik (penting untuk laporan):** primary metric sempat direvisi dua kali
selama eksperimen. Awalnya MCC, lalu dipivot ke F1-Score (weighted) supaya sebanding langsung
dengan paper, lalu DIKEMBALIKAN ke MCC (revisi final) setelah disadari F1-weighted rawan bias:
kalau primary metric F1-weighted dipakai, Improvement B (E2) tampak "nyaris netral/positif" (F1-w
naik tipis +0,0003 di CV) padahal MCC menunjukkan E2 tetap NET LEBIH BURUK dari baseline
(-0,0010) -- perbedaan ini murni karena F1-weighted didominasi kelas mayoritas dan tidak menghukum
trade-off precision-recall secara adil, sementara MCC (satu skalar dari confusion matrix penuh)
konsisten menunjukkan arah yang sama di semua kandidat. MCC dipilih sebagai kriteria FINAL karena
tidak bisa "dipilih-pilih" berdasar hasil siapa yang menang -- keputusan metrik harus dibuat
sebelum melihat hasil, bukan sesudahnya.

**Apakah hasil eksperimen mendukung hipotesis awal?**

Tidak. Hipotesis Improvement A (log-transform akan membuat StandardScaler lebih representatif dan
membantu model membedakan pola) TIDAK terbukti -- MCC malah turun (-0,0089 di CV train-only).
Kemungkinan penyebab: XGBoost sebagai model berbasis pohon TIDAK sensitif terhadap monotonic
transformation seperti log1p -- keputusan split tree hanya bergantung pada urutan/ranking nilai,
bukan skala absolutnya, sehingga log-transform tidak menambah informasi baru bagi model, hanya
mengubah skala yang secara teori tidak relevan untuk tree-based model. Manfaat log-transform lebih
terasa untuk model berbasis jarak/linear (KNN, regresi linear, SVM dengan kernel linear) yang
sensitif terhadap skala fitur -- bukan untuk XGBoost.

Hipotesis Improvement B (drop fitur redundan akan netral-atau-sedikit membantu) sebagian benar:
efeknya memang kecil dan konsisten negatif tipis di MCC (-0,0010 CV, -0,0004 test set) -- "nyaris
netral" adalah deskripsi yang tepat, tapi tetap bukan perbaikan.

**Berapa banyak perbaikan dari preprocessing? Berapa banyak dari tuning?**

Di test set, dari Baseline Terkoreksi ke Improved Preprocessing (E3): MCC TURUN 0,0004 --
preprocessing (kombinasi A+B) memberi dampak negatif kecil, bukan perbaikan.
Dari Improved Preprocessing ke Final Tuned: MCC TURUN LAGI 0,0111 -- tuning tambahan memperburuk
hasil secara lebih substansial dibanding efek preprocessing. Kemungkinan penyebab: hyperparameter
"terbaik" di CV train (max_depth=3, learning_rate=0.1, n_estimators=100, MCC CV=0,6358) overfit
terhadap struktur fold train dan tidak generalize sama baiknya ke test set yang belum pernah
dilihat -- gap CV-vs-test (0,6358 vs 0,5840, selisih 0,0518) cukup besar untuk dicurigai sebagai
overfitting terhadap CV, bukan perbaikan generalisasi nyata.
Kesimpulan: baik preprocessing maupun tuning TIDAK memberi perbaikan nyata dibanding Baseline
Terkoreksi untuk pipeline XGBoost pada dataset ini -- ini adalah hasil eksperimen yang sah dan
informatif, bukan kegagalan proses.

**Kemungkinan penyebab hasil ini:**

- **Log-transform tidak relevan untuk model tree-based.** XGBoost membangun split berdasarkan
  threshold pada nilai fitur, dan urutan relatif nilai (yang menentukan split mana yang optimal)
  tidak berubah oleh transformasi monotonic seperti log1p. Transformasi ini lebih berguna untuk
  model yang sensitif terhadap skala/jarak Euclidean (KNN, SVM, regresi linear, jaringan saraf
  dengan gradient descent) -- bukan untuk pohon keputusan atau ensemble pohon.
- **Fitur redundan tidak banyak mengganggu XGBoost.** Model berbasis pohon relatif robust terhadap
  multikolinearitas -- drop salah satu dari pasangan fitur berkorelasi tinggi hanya menghilangkan
  informasi tanpa menghilangkan noise yang berarti bagi algoritma ini.
- **Hyperparameter "terbaik" di CV tidak generalize ke test set.** Gap CV-vs-test yang cukup besar
  (0,0518 poin MCC) mengindikasikan search space kecil (27 kombinasi) dengan std CV yang relatif
  besar (0,02-0,03) rentan menghasilkan kandidat yang overfit terhadap struktur spesifik 5-fold
  yang dipakai, bukan menemukan konfigurasi yang benar-benar generalize lebih baik.
- **Baseline paper (chi2 top-20 + SMOTE 1:1 + max_depth=2) sudah relatif dekat optimal** untuk
  dataset dan model ini -- konsisten dengan temuan notebook 03 (replikasi murni) yang menunjukkan
  bahkan tanpa koreksi anti-leakage sekalipun, hasilnya sudah tidak jauh dari versi terkoreksi.
  Ruang untuk perbaikan lewat preprocessing tambahan atau tuning ringan tampaknya memang terbatas.

**Konteks tambahan -- progresi 3 tahap (replikasi murni -> terkoreksi -> improvement), dilihat
lewat metrik weighted supaya sebanding dengan paper:**

| Tahap | Precision (weighted) | F1 (weighted) | MCC |
|---|---|---|---|
| Replikasi Murni dari Paper | 0,8936 | 0,8875 | 0,5898 |
| Baseline Terkoreksi (starting point Tugas #3) | 0,8951 | 0,8892 | 0,5955 |
| Final Tuned Pipeline (hasil Tugas #3) | 0,8936 | 0,8808 | 0,5840 |
| **Paper** | **0,9001** | **0,898** | -- |

Koreksi metodologi Tugas #2 (fit chi2+scaling hanya di train) tetap memberi kenaikan kecil
dibanding replikasi murni (+0,0015 Precision-w, +0,0017 F1-w, +0,0057 MCC) -- konsisten dengan
temuan notebook 03. Perbaikan pipeline Tugas #3 (log-transform + feature pruning + tuning) justru
menjauh dari paper di semua metrik dibanding Baseline Terkoreksi. Kesimpulan jujur: untuk model
XGBoost pada dataset ini, koreksi anti-leakage Tugas #2 memberi kontribusi kecil namun positif,
sementara upaya perbaikan lebih lanjut di Tugas #3 (2 kategori improvement resmi brief yang
dieksplorasi) tidak berhasil mendekatkan hasil ke paper lebih jauh -- baseline terkoreksi tetap
titik terbaik yang ditemukan sejauh ini.

**Limitasi:**

- Hanya satu model (XGBoost) yang diuji secara mendalam sesuai batasan brief (tidak ganti model).
  Log-transform kemungkinan akan memberi hasil berbeda (mungkin membantu) untuk model yang lebih
  sensitif skala seperti SVM atau MLP -- di luar scope karena brief meminta model utama tetap sama.
- Search space hyperparameter kecil (27 kombinasi) dengan gap CV-vs-test yang cukup besar
  menunjukkan hasil tuning kurang meyakinkan secara statistik tanpa uji signifikansi formal (mis.
  nested cross-validation atau paired t-test antar fold).
- Hanya dua dari 8 kategori improvement resmi brief yang dieksplorasi secara formal (numerical
  transformations, removal of redundant features) -- kategori lain (missing-value handling tidak
  relevan karena dataset tidak punya missing value; categorical encoding, scaling/normalization,
  outlier handling, feature engineering, feature selection lain belum dicoba secara formal)
  mungkin memberi hasil berbeda jika dieksplorasi.
- Perbedaan kecil antar eksperimen (E0 vs E2, misalnya) berada dalam rentang std CV yang terukur,
  sehingga signifikansi statistiknya tidak diuji formal.

## Ringkasan File Output

- `results/tables/tugas3_e0_e3_comparison.csv` -- perbandingan E0-E3 (train-only CV)
- `results/tables/tugas3_gridsearch_top10.csv` -- top-10 kombinasi hyperparameter (train-only CV)
- `results/tables/tugas3_final_test_evaluation.csv` -- evaluasi akhir 4 pipeline di test set
